# Botnet Detection: Generalization Test (Cross-Dataset)

This notebook is the final 'Proof of Validity' for our model. We have achieved 99% accuracy on the CIC-DDoS2019 dataset, but the real question is:

**Can a model trained on one network (CIC) detect botnets from a COMPLETELY different environment (Sample CSV)?**

### Objectives:
1. **Load Unseen Data**: Use `botnet_sample.csv` (likely BoT-IoT or CTU-13 data).
2. **Feature Mapping**: Convert a 5-feature schema into the 22-feature schema our model expects.
3. **Zero-Shot Test**: Run the trained model on this external data without any new training.
4. **Analyze Generalization**: Does the model truly understand 'Botnet Behavior'?

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os

print('Generalization test environment ready.')

Generalization test environment ready.


## 1. Load the 'External' Dataset

We use `botnet_sample.csv`. This data has different columns and different timing units than our training data.

In [6]:
df_ext = pd.read_csv('data/botnet_sample.csv')
print(f'External Data Samples: {len(df_ext)}')
print(df_ext.head())
print('\nLabels (0=Normal, 1=Botnet):')
print(df_ext['label'].value_counts())

External Data Samples: 2000
        dur        sbytes         dbytes  spkts  dpkts proto state  label
0  0.563122    514.931986    5963.542455     26      6   udp   REQ      0
1  3.612146    570.343774  452758.026162      5      1  icmp   CON      0
2  1.580095   1895.129051      75.176938     13      5   tcp   CON      0
3  1.095531   6215.062211    2794.584672      1      9   tcp   CON      0
4  0.040710  55591.274870   10733.080323     15     30   tcp   CON      1

Labels (0=Normal, 1=Botnet):
label
0    1689
1     311
Name: count, dtype: int64


## 2. Feature Mapping (The Bridge)

Our model expects 22 features. The CSV only has 5 primary ones. We must map them carefully and handle unit conversions (e.g., Seconds to Microseconds).

In [7]:
def map_to_cic_schema(df):
    new_df = pd.DataFrame(index=df.index)
    
    # Main behavior mapping
    new_df['Flow Duration'] = df['dur'] * 1000000  # Sec to Microseconds
    new_df['Total Fwd Packets'] = df['spkts']
    new_df['Total Backward Packets'] = df['dpkts']
    new_df['Fwd Packets Length Total'] = df['sbytes']
    new_df['Bwd Packets Length Total'] = df['dbytes']
    
    # Calculated features
    # Avoid division by zero
    duration_safe = df['dur'].replace(0, 0.000001)
    pkts_total_safe = (df['spkts'] + df['dpkts']).replace(0, 1)
    
    new_df['Flow Bytes/s'] = (df['sbytes'] + df['dbytes']) / duration_safe
    new_df['Flow Packets/s'] = (df['spkts'] + df['dpkts']) / duration_safe
    new_df['Fwd Packets/s'] = df['spkts'] / duration_safe
    new_df['Bwd Packets/s'] = df['dpkts'] / duration_safe
    
    new_df['Packet Length Mean'] = (df['sbytes'] + df['dbytes']) / pkts_total_safe
    new_df['Avg Packet Size'] = new_df['Packet Length Mean']
    new_df['Down/Up Ratio'] = df['dpkts'] / df['spkts'].replace(0, 1)
    
    # Protocol mapping
    proto_map = {'tcp': 6, 'udp': 17, 'icmp': 1}
    new_df['Protocol'] = df['proto'].str.lower().map(proto_map).fillna(0)
    
    # Filling unknown features with zeros (since we don't have flags/window info in CSV)
    # NOTE: This tests if the model is robust enough to predict without 'cheats'!
    unknown_feats = [
        'Flow IAT Mean', 'Flow IAT Std', 'Fwd IAT Total', 'Bwd IAT Total',
        'Packet Length Std', 'SYN Flag Count', 'ACK Flag Count', 'RST Flag Count',
        'Init Fwd Win Bytes'
    ]
    for f in unknown_feats:
        new_df[f] = 0
        
    return new_df

X_ext = map_to_cic_schema(df_ext)
print(f'Mapped External Data Shape: {X_ext.shape}')

Mapped External Data Shape: (2000, 22)


## 3. Zero-Shot Testing

We load our saved model from `results/botnet_classifier.joblib` and evaluate it on this new data.

In [8]:
if not os.path.exists('results/botnet_classifier.joblib'):
    print('[!] Model file not found. Please run main.py or the main notebook first.')
else:
    # Load artifact
    checkpoint = joblib.load('results/botnet_classifier.joblib')
    model = checkpoint['model']
    scaler = checkpoint['scaler']
    features = checkpoint['features']
    
    # Ensure column order matches training
    X_ext_final = X_ext[features]
    X_ext_scaled = scaler.transform(X_ext_final)
    
    # Predict
    y_pred_multi = model.predict(X_ext_scaled)
    
    # Map multi-class to binary for comparison with CSV labels (Label 1 = Attack)
    # In our model: Benign=0, Syn=..., LDAP=..., UDP=... (All others > 0 are attacks)
    y_pred_binary = (y_pred_multi > 0).astype(int)
    y_true_binary = df_ext['label']
    
    print(f'Accuracy on External Data: {accuracy_score(y_true_binary, y_pred_binary):.4f}')
    print('\nExternal Classification Report (Binary):')
    print(classification_report(y_true_binary, y_pred_binary))

Accuracy on External Data: 0.8095

External Classification Report (Binary):
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      1689
           1       0.09      0.03      0.04       311

    accuracy                           0.81      2000
   macro avg       0.47      0.49      0.47      2000
weighted avg       0.73      0.81      0.76      2000



## 4. Analysis & Conclusion

If the accuracy is high (e.g. >80%), it means our model is **valid** and has learned universal network botnet patterns.

If the accuracy is low, it means our model is **overfitted** to the CIC dataset. In that case, we should proceed with 'Mixing' datasets in a future training run to create a more 'Universal Botnet Brain'.